In [3]:
!pip install chromadb
!pip install wikipedia

In [4]:
from transformers import pipeline
import wikipedia
wikipedia.set_user_agent('WikiRAG (Colab; Educational Use)')
import chromadb
from chromadb.utils import embedding_functions
from google.colab import drive

drive.mount('/content/drive')

# database setup
db_path = '/content/drive/MyDrive/wikipedia_db'
database = chromadb.PersistentClient(path=db_path)

# setup embed function and collection for database
embed_fun = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = database.get_or_create_collection(name="wiki_search", embedding_function=embed_fun)

# load small chatbot
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
chatbot = pipeline('text-generation', model=MODEL_NAME, dtype='auto', device_map='auto', trust_remote_code=True)
chatbot.generation_config.max_new_tokens = 256
chatbot.generation_config.max_length = None


# text chunker
def chunk_text(text, chunk_size=500):
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        if current_length + len(word) + 1 > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = []
            current_length = 0
        current_chunk.append(word)
        current_length += len(word) + 1

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks


def fetch_and_store_article(search_query):
    try:
        # Search Wikipedia, fetch top result
        results = wikipedia.search(search_query, results=5)
        if not results:
            print("No Wikipedia articles found for that query.")
            return None

        page = wikipedia.page(results[0], auto_suggest=False)

    except wikipedia.DisambiguationError as e:
        # If result is a disambiguation page, pick first option
        try:
            page = wikipedia.page(e.options[0], auto_suggest=False)
        except Exception:
            print(f"Could not resolve disambiguation for '{search_query}'.")
            return None
    except wikipedia.PageError:
        print(f"Wikipedia page not found for '{search_query}'.")
        return None
    except Exception as e:
        print(f"Error fetching Wikipedia article: {e}")
        return None

    # Chunk and store article in ChromaDB
    chunks = chunk_text(page.content)
    if not chunks:
        print("Article had no content to index.")
        return None

    ids = [f"{page.title}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"title": page.title, "chunk_index": i} for i in range(len(chunks))]

    collection.upsert(
        documents=chunks,
        ids=ids,
        metadatas=metadatas,
    )

    print(f"Indexed {len(chunks)} chunks from Wikipedia article: '{page.title}'")
    return page.title


# Search and retrieve loop
while True:
    search_query = input("\n\n=+=+=+=+=+=+=+=+=+=+=+=+=\nSearch Wikipedia: ")
    print("=+=+=+=+=+=+=+=+=+=+=+=+=")

    # Fetch article from Wikipedia and store it in ChromaDB
    article_title = fetch_and_store_article(search_query)
    if article_title is None:
        print("Failed to fetch article. Please try a different query.\n")
        continue

    # Retrieve most relevant chunks from database
    retrieved_results = collection.query(
        query_texts=[search_query],
        n_results=3
    )

    # If no results returned, prompt again
    if not retrieved_results.get('distances') or not retrieved_results['distances'][0]:
        print("Failed to find any related articles. Please try a different query.\n")
        continue

    # Check distance of best match.
    # Lower is better. If distance > 1.5, the content is likely not relevant enough.
    best_distance = retrieved_results['distances'][0][0]
    if best_distance > 1.5:
        print(f"Failed to find a highly relevant article (Match distance: {best_distance:.2f}). Please try a more specific query.\n")
        continue

    article_name = "Unknown Article"  # fallback if title and id are missing
    # Get title from metadata (fallback to ID)
    article_name = retrieved_results['metadatas'][0][0].get('title') or retrieved_results['ids'][0][0]

    print(f"\nFound relevant article: {article_name}")
    print(f"[Retrieved {len(retrieved_results['documents'][0])} snippets about '{search_query}']")

    # Save retrieved context and break loop
    retrieved_context = "\n\n".join(retrieved_results['documents'][0])
    break

# Conversation Loop
conversation_history = [
    {
        'role': 'system',
        'content': (
            'You are a helpful wikipedia research assistant. '
            'Do NOT use any markdown in your output; output should be plain text only. '
            'Answer the question using the provided Wikipedia context; if the answer is not there, say you don\'t know. '
            'Do not make any references in your response to the above text or the phrase "Wikipedia context".\n\n'
            f'Wikipedia context:\n{retrieved_context}'
        ),
    },
]

print(f"\n(Type '/quit' or '/exit' to end the conversation)\n")

while True:
    question = input(f"\n~x~x~x~x~x~x~x~x~x~x~x~x~\nQuestion about '{article_name}': ")
    print("~x~x~x~x~x~x~x~x~x~x~x~x~")

    if question.strip().lower() in ('/quit', '/exit', '\exit', '\quit'):
        print("\n~o~o~o~o~o~o~o~o~o~o~o~o~\nQwen: Goodbye!\n~o~o~o~o~o~o~o~o~o~o~o~o~")
        break

    # Add question to conversation history
    conversation_history.append({
        'role': 'user',
        'content': question,
    })

    response = chatbot(conversation_history)

    # Extract assistant's reply from response
    assistant_reply = response[0]['generated_text'][-1]['content']

    # Add assistant's reply to conversation history for continuity
    conversation_history.append({
        'role': 'assistant',
        'content': assistant_reply,
    })

    print(f"\n~o~o~o~o~o~o~o~o~o~o~o~o~\nQwen: {assistant_reply}")
    print("~o~o~o~o~o~o~o~o~o~o~o~o~\n")

<>:149: SyntaxWarning: invalid escape sequence '\e'
<>:149: SyntaxWarning: invalid escape sequence '\q'
<>:149: SyntaxWarning: invalid escape sequence '\e'
<>:149: SyntaxWarning: invalid escape sequence '\q'
/tmp/ipykernel_519/1158504047.py:149: SyntaxWarning: invalid escape sequence '\e'
  if question.strip().lower() in ('/quit', '/exit', '\exit', '\quit'):
/tmp/ipykernel_519/1158504047.py:149: SyntaxWarning: invalid escape sequence '\q'
  if question.strip().lower() in ('/quit', '/exit', '\exit', '\quit'):


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]



=+=+=+=+=+=+=+=+=+=+=+=+=
Search Wikipedia: Fruit in Summer
=+=+=+=+=+=+=+=+=+=+=+=+=
Indexed 5 chunks from Wikipedia article: 'Tree of 40 Fruit'

Found relevant article: Tree of 40 Fruit
[Retrieved 3 snippets about 'Fruit in Summer']

(Type '/quit' or '/exit' to end the conversation)


~x~x~x~x~x~x~x~x~x~x~x~x~
Question about 'Tree of 40 Fruit': What is the tree of 40 fruit?
~x~x~x~x~x~x~x~x~x~x~x~x~

~o~o~o~o~o~o~o~o~o~o~o~o~
Qwen: The Tree of 40 Fruit is a unique fruit tree created by Sam Van Aken, an associate professor of sculpture at Syracuse University. Using the technique of grafting, the tree combines branches from forty different "donor" trees, each producing a different variety of stone fruit such as almond, apricot, cherry, nectarine, peach, and plum. The tree blooms each spring with a mix of red, pink, and white flowers and produces fruit of various types that ripen sequentially from July to October in the United States. Originally conceived as an art project, it also hi

## Long Creative Synthesis

###WikiRAG
searches for a related wikipedia article and retrieves information from the article to answer user questions. It has persistent storage via ChromaDB so if you prompt it again about an article it already has saved, it saves time from having to refind the article. I want to eventually implement a way for it to find an article from the users questions alone and to use the persistent storage to be able to prompt about multiple saved articles at the same time, but implementation seems difficult and it is always better to K.I.S.S. (Keep it simple stupid).